# Experiment 2.2d: Is Muon Just Implicit Per-Layer Learning Rate Adaptation?

---

## Theoretical Background

The Newton-Schulz orthogonalization in Muon has a subtlety that is easy to overlook:
before iterating, it normalizes the input matrix by its Frobenius norm
$X_0 = M / \|M\|_F$. This means that regardless of how large or small a layer's
gradient (or momentum) is, the orthogonalized output always has $O(1)$ spectral norm.

This is equivalent to an **implicit per-layer learning rate**: layers with large
gradients get their updates scaled down, while layers with small gradients get scaled up.
Under rescaling $W_0 \to cW_0,\; W_{L-1} \to W_{L-1}/c$, the gradient norms change
dramatically across layers, but Muon's NS normalization absorbs this automatically.

## The Critical Question

**Is per-layer LR adaptation the ENTIRE explanation for Muon's advantage, or does
the directional content (polar factor / SV equalization) provide additional value?**

To answer this, we construct an **oracle competitor**: SGD with independently tuned
learning rates for each layer, optimized by exhaustive grid search over $5^4 = 625$
combinations. This oracle has access to more information than any practical optimizer
would -- if it still cannot match Muon, then Muon's advantage must come from the
*direction* of the update, not just its *scale*.

## Experimental Design

| Method | LR Tuning | Gradient Processing |
|:-------|:----------|:-------------------|
| SGD (single LR) | 1 global LR from 7 candidates | Raw gradients |
| SGD (oracle per-layer) | 4 independent LRs from $5^4=625$ combos | Raw gradients |
| Muon (single LR) | 1 global LR from 7 candidates | Newton-Schulz orthogonalized |

All methods use momentum ($\beta = 0.9$) and are evaluated at $c = 100$ rescaling.

## Key Test

| Test | Criterion | Implication |
|:-----|:----------|:------------|
| T1 | Oracle SGD matches Muon within 2x loss | Muon = implicit per-layer LR (no directional value) |
| T1 fails | Oracle SGD > 2x Muon's loss | Muon provides directional value beyond LR scaling |

## Environment Setup

Import NumPy for linear algebra and itertools for the exhaustive grid search over
per-layer learning rate combinations.

In [ ]:
import numpy as np
import os
import itertools

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

**Network**: 4-layer deep linear, 32x32, with $c = 100$ rescaling applied to the first
and last layers. This creates a 4-order-of-magnitude spread in per-layer weight norms.

**Training**: 300 steps with momentum $\beta = 0.9$. We use 5 seeds total: the first 3
for LR selection (to avoid overfitting the LR to a single data realization) and all 5
for final evaluation.

In [ ]:
DIM = 32
N_LAYERS = 4
NUM_STEPS = 300
MOMENTUM_BETA = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64
C = 100

print("\n--- Experimental Configuration ---")
print(f"  DIM = {DIM}")
print(f"  NUM_STEPS = {NUM_STEPS}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  NS_ITERS = {NS_ITERS}")
print(f"  N_LAYERS = {N_LAYERS}")


In [ ]:
# 7 candidates for single-LR methods
SGD_LR_CANDIDATES = [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]
MUON_LR_CANDIDATES = [0.05, 0.03, 0.02, 0.01, 0.007, 0.005, 0.003]

### Learning Rate Search Spaces

The single-LR candidates for SGD and Muon span different ranges because Muon's NS
normalization changes the effective update scale. The per-layer grid uses 5 values
spanning 4 orders of magnitude ($10^{-1}$ to $10^{-5}$), deliberately covering the
full range of scales created by the $c=100$ rescaling. With 4 layers and 5 candidates
each, this gives $5^4 = 625$ combinations -- exhaustive but computationally feasible.

In [ ]:
# 5 candidates per layer for oracle grid search (5^4 = 625 combos)
PER_LAYER_LR_CANDIDATES = [0.1, 0.01, 0.001, 0.0001, 0.00001]

## Core Algorithms

### Newton-Schulz Orthogonalization

The key operation in Muon. The Frobenius normalization $X_0 = M/\|M\|_F$ is
the source of the implicit per-layer LR: it erases the scale of the input,
leaving only directional information for the subsequent NS iterations.

### Weight Initialization with Rescaling

Near-identity initialization ($W = I + 0.1\epsilon$) then rescaled:
$W_0 \to 100 \cdot W_0$ and $W_3 \to 0.01 \cdot W_3$. This preserves
the product $W_3 W_2 W_1 W_0$ but creates extreme gradient norm asymmetry.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def init_weights(seed, c):
    rng = np.random.RandomState(seed)
    weights = []
    for l in range(N_LAYERS):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W)
    # Apply c-rescaling: layer 0 *= c, layer -1 *= 1/c
    weights[0] = weights[0] * c
    weights[-1] = weights[-1] / c
    return weights

## Network Forward Pass and Loss

Standard deep linear forward pass $\hat{Y} = W_3 W_2 W_1 W_0 X$ with MSE loss.
The data is generated from a random target matrix $W^* \sim \mathcal{N}(0, 0.5^2)$
so the optimal solution is $W_3 \cdots W_0 = W^*$.

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

### Data Generation

We generate supervised pairs $(X, Y)$ where $Y = W^* X$ for a random target matrix
$W^*$. This means the optimal product $W_3 W_2 W_1 W_0$ should approximate $W^*$.
The factorization into 4 layers is not unique -- the deep linear network must find
one specific factorization, and how it distributes the spectral content across layers
depends on the optimizer.

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    W_target = rng.randn(DIM, DIM) * 0.5
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = W_target @ X
    return X, Y

## Training Methods

Three training functions implement the three competitors:

1. **SGD (single LR)**: Classical SGD with momentum. Uses one global learning rate.
   Under $c=100$ rescaling, this struggles because the gradient norms differ by orders
   of magnitude across layers -- a single LR cannot be optimal for all of them.

2. **Muon (single LR)**: Applies Newton-Schulz to each layer's gradient before
   accumulating into momentum. The NS normalization acts as an automatic per-layer
   scale equalizer.

3. **SGD (oracle per-layer LR)**: SGD with independently chosen LR per layer. This
   is the "unfair" baseline -- it has access to the optimal LR for each layer via
   exhaustive grid search. If Muon is *just* per-layer LR adaptation, this oracle
   should match it.

In [ ]:
def train_sgd(weights_init, X, Y, lr):
    """SGD with momentum, single global LR."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for l in range(N_LAYERS):
            mom[l] = MOMENTUM_BETA * mom[l] + grads[l]
            weights[l] = weights[l] - lr * mom[l]
    return compute_loss(weights, X, Y)

In [ ]:
def train_muon(weights_init, X, Y, lr):
    """Muon: orthogonalize gradient via Newton-Schulz, then momentum."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for l in range(N_LAYERS):
            mom[l] = MOMENTUM_BETA * mom[l] + newton_schulz(grads[l])
            weights[l] = weights[l] - lr * mom[l]
    return compute_loss(weights, X, Y)

In [ ]:
def train_sgd_per_layer(weights_init, X, Y, per_layer_lrs):
    """SGD with momentum, independent LR per layer."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for l in range(N_LAYERS):
            mom[l] = MOMENTUM_BETA * mom[l] + grads[l]
            weights[l] = weights[l] - per_layer_lrs[l] * mom[l]
    return compute_loss(weights, X, Y)

## Learning Rate Sweep Utilities

### Single-LR Sweep
Evaluates each candidate LR on the first 3 seeds and selects the best.
Using a subset of seeds for selection prevents overfitting the hyperparameter
to one specific data realization.

### Oracle Per-Layer LR Grid Search
Exhaustively evaluates all $5^4 = 625$ combinations of per-layer learning rates.
This is computationally expensive but provides the strongest possible baseline:
if SGD with this oracle cannot match Muon, then Muon's advantage is fundamentally
about update *direction*, not *scale*.

In [ ]:
def sweep_single_lr(train_fn, seeds, candidates):
    """Sweep over LR candidates using first 3 seeds. Return best LR."""
    best_lr, best_loss = candidates[-1], float('inf')
    for lr in candidates:
        losses = []
        for s in seeds[:3]:
            X, Y = make_data(s)
            w = init_weights(s + 5000, C)
            fl = train_fn(w, X, Y, lr)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        ml = np.mean(finite) if finite else float('inf')
        if ml < best_loss:
            best_loss = ml
            best_lr = lr
    return best_lr, best_loss

In [ ]:
def sweep_oracle_per_layer(seeds):
    """
    Grid search: 5 LRs per layer, 5^4 = 625 combos.
    Use first 3 seeds for selection. Return best per-layer LR tuple.
    """
    combos = list(itertools.product(PER_LAYER_LR_CANDIDATES, repeat=N_LAYERS))
    print(f"    Oracle grid search: {len(combos)} combos...")

    best_combo = None
    best_loss = float('inf')

    for idx, combo in enumerate(combos):
        lrs = list(combo)
        losses = []
        for s in seeds[:3]:
            X, Y = make_data(s)
            w = init_weights(s + 5000, C)
            fl = train_sgd_per_layer(w, X, Y, lrs)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        ml = np.mean(finite) if finite else float('inf')
        if ml < best_loss:
            best_loss = ml
            best_combo = lrs

        if (idx + 1) % 100 == 0:
            print(f"      {idx+1}/{len(combos)} combos evaluated, best so far: {best_loss:.6e}")

    return best_combo, best_loss

## Main Experiment: Three-Way Comparison

The experiment proceeds in two phases:

**Phase 1 -- LR Selection**: Using the first 3 seeds, find the best single LR for
SGD and Muon, and the best per-layer LR tuple for oracle SGD. This uses
$3 \times (7 + 7 + 625) = 1917$ training runs.

**Phase 2 -- Full Evaluation**: Using all 5 seeds with the selected LRs, compute
the final loss for each method. The 2 held-out seeds provide a form of
generalization check.

The critical diagnostic is comparing gradient norms at initialization with the
oracle-selected per-layer LRs. If the oracle LRs are inversely proportional to
gradient norms, it confirms that per-layer LR adaptation is compensating for
the scale imbalance created by the $c = 100$ rescaling.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print(f"2.2d: MUON AS IMPLICIT PER-LAYER LR (c={C})")
    print("=" * 100)
    print(f"Network: {N_LAYERS}-layer deep linear, {DIM}x{DIM}, {NUM_STEPS} steps")
    print(f"Rescaling: layer 0 *= {C}, layer {N_LAYERS-1} *= {1.0/C}")
    print(f"SGD LR candidates:  {SGD_LR_CANDIDATES}")
    print(f"Muon LR candidates: {MUON_LR_CANDIDATES}")
    print(f"Per-layer LR grid:  {PER_LAYER_LR_CANDIDATES} (5^4 = {len(PER_LAYER_LR_CANDIDATES)**N_LAYERS} combos)")
    print()

    # Phase 1: LR sweeps
    print("Phase 1: LR sweeps...")

    print("  SGD single LR sweep...")
    sgd_lr, sgd_sweep_loss = sweep_single_lr(train_sgd, seeds, SGD_LR_CANDIDATES)
    print(f"    Best: lr={sgd_lr}, sweep loss={sgd_sweep_loss:.6e}")

    print("  Muon single LR sweep...")
    muon_lr, muon_sweep_loss = sweep_single_lr(train_muon, seeds, MUON_LR_CANDIDATES)
    print(f"    Best: lr={muon_lr}, sweep loss={muon_sweep_loss:.6e}")

    print("  SGD oracle per-layer LR sweep...")
    oracle_lrs, oracle_sweep_loss = sweep_oracle_per_layer(seeds)
    print(f"    Best per-layer LRs: {oracle_lrs}")
    print(f"    Sweep loss: {oracle_sweep_loss:.6e}")

    # Phase 2: Full training with all seeds
    print(f"\nPhase 2: Full training ({NUM_SEEDS} seeds)...")

    results = {}
    for name, desc in [('sgd', 'SGD (single LR)'),
                       ('muon', 'Muon (single LR)'),
                       ('sgd_oracle', 'SGD (oracle per-layer LR)')]:
        losses = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000, C)
            if name == 'sgd':
                fl = train_sgd(w, X, Y, sgd_lr)
            elif name == 'muon':
                fl = train_muon(w, X, Y, muon_lr)
            elif name == 'sgd_oracle':
                fl = train_sgd_per_layer(w, X, Y, oracle_lrs)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        mean_loss = np.mean(finite) if finite else float('inf')
        std_loss = np.std(finite) if len(finite) > 1 else 0.0
        finite_frac = len(finite) / len(losses) * 100
        results[name] = {
            'mean': mean_loss,
            'std': std_loss,
            'finite_frac': finite_frac,
            'losses': losses,
        }
        print(f"  {desc:>30}: loss={mean_loss:.6e} +/- {std_loss:.6e}  (finite={finite_frac:.0f}%)")

    # ==========================================================================
    # RESULTS TABLE
    # ==========================================================================

    muon_loss = results['muon']['mean']
    sgd_loss = results['sgd']['mean']
    oracle_loss = results['sgd_oracle']['mean']

    print(f"\n\n{'=' * 100}")
    print("RESULTS")
    print(f"{'=' * 100}")

    print(f"\n  {'Method':>30}  {'Loss':>14}  {'Std':>14}  {'vs Muon':>10}  {'Finite%':>8}  {'LR(s)':>20}")
    print("  " + "-" * 100)
    print(f"  {'SGD (single LR)':>30}  {sgd_loss:>14.6e}  {results['sgd']['std']:>14.6e}  "
          f"{sgd_loss/max(muon_loss,1e-30):>10.2f}x  {results['sgd']['finite_frac']:>7.0f}%  lr={sgd_lr}")
    print(f"  {'Muon (single LR)':>30}  {muon_loss:>14.6e}  {results['muon']['std']:>14.6e}  "
          f"{'1.00x':>10}  {results['muon']['finite_frac']:>7.0f}%  lr={muon_lr}")
    print(f"  {'SGD (oracle per-layer)':>30}  {oracle_loss:>14.6e}  {results['sgd_oracle']['std']:>14.6e}  "
          f"{oracle_loss/max(muon_loss,1e-30):>10.2f}x  {results['sgd_oracle']['finite_frac']:>7.0f}%  "
          f"lrs={[f'{l:.1e}' for l in oracle_lrs]}")

    # Show gradient norms at init to explain why per-layer LR is needed
    print(f"\n  Gradient norms at init (c={C} rescaling):")
    X, Y = make_data(seeds[0])
    w = init_weights(seeds[0] + 5000, C)
    grads = compute_gradients(w, X, Y)
    for l in range(N_LAYERS):
        g_norm = np.linalg.norm(grads[l], 'fro')
        print(f"    Layer {l}: ||G||_F = {g_norm:.4e}  (oracle LR = {oracle_lrs[l]:.1e})")

    # ==========================================================================
    # HYPOTHESIS TESTS
    # ==========================================================================

    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    # T1: Oracle per-layer SGD matches Muon within 2x?
    ratio_oracle = oracle_loss / max(muon_loss, 1e-30)
    t1 = ratio_oracle < 2.0
    print(f"\n  T1: SGD with ORACLE per-layer LR matches Muon within 2x?")
    print(f"      Oracle SGD loss:  {oracle_loss:.6e}")
    print(f"      Muon loss:        {muon_loss:.6e}")
    print(f"      Ratio:            {ratio_oracle:.2f}x")
    if t1:
        print(f"      --> PASS: Muon's advantage IS explained by implicit per-layer LR.")
        print(f"         With perfect per-layer tuning, SGD matches Muon.")
    else:
        print(f"      --> FAIL: Muon provides DIRECTIONAL value beyond LR scaling.")
        print(f"         Even with oracle per-layer LR, SGD cannot match Muon.")

    # How much does single-LR SGD suffer?
    ratio_sgd = sgd_loss / max(muon_loss, 1e-30)
    print(f"\n  Context: SGD (single LR) vs Muon = {ratio_sgd:.1f}x")
    print(f"  The per-layer LR reduces the gap from {ratio_sgd:.1f}x to {ratio_oracle:.2f}x")
    if ratio_sgd > 1 and ratio_oracle > 1:
        pct_explained = (1 - (ratio_oracle - 1) / (ratio_sgd - 1)) * 100 if ratio_sgd > 1 else 0
        print(f"  Per-layer LR explains {pct_explained:.0f}% of Muon's advantage")
    elif ratio_oracle <= 1:
        print(f"  Per-layer LR FULLY explains Muon's advantage")

    # ==========================================================================
    # CONCLUSION
    # ==========================================================================

    print(f"\n\n{'=' * 100}")
    print("CONCLUSION")
    print(f"{'=' * 100}")

    if t1:
        print(f"\n  Muon's robustness to c={C} rescaling IS primarily per-layer LR adaptation.")
        print(f"  SGD with oracle per-layer LR achieves {ratio_oracle:.2f}x Muon's loss.")
        print(f"  The Newton-Schulz normalization acts as an automatic per-layer LR tuner.")
    else:
        print(f"\n  Muon provides value BEYOND per-layer LR adaptation.")
        print(f"  Even with oracle per-layer LR (625 combos searched), SGD achieves")
        print(f"  only {ratio_oracle:.2f}x Muon's loss. The directional quality of the")
        print(f"  polar factor (SV equalization) provides additional convergence benefit")
        print(f"  that cannot be replicated by LR tuning alone.")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

    # ==========================================================================
    # PLOT
    # ==========================================================================
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f'2.2d: Muon as Implicit Per-Layer LR (c={C})\n'
                     f'{N_LAYERS}-layer deep linear {DIM}x{DIM}, {NUM_STEPS} steps',
                     fontsize=13, fontweight='bold')

        # (a) Loss comparison bar chart
        ax = axes[0]
        methods = ['SGD\n(single LR)', 'SGD\n(oracle per-layer)', 'Muon\n(single LR)']
        losses_plot = [sgd_loss, oracle_loss, muon_loss]
        stds_plot = [results['sgd']['std'], results['sgd_oracle']['std'], results['muon']['std']]
        colors = ['#4477AA', '#FF8800', '#CC3311']
        bars = ax.bar(range(3), losses_plot, yerr=stds_plot, color=colors,
                      edgecolor='black', capsize=5, width=0.6)
        ax.set_xticks(range(3))
        ax.set_xticklabels(methods, fontsize=9)
        ax.set_ylabel('Final Loss')
        ax.set_yscale('log')
        ax.set_title(f'Final Loss (c={C} rescaling)')
        ax.grid(True, alpha=0.3, axis='y')
        for i, bar in enumerate(bars):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                    f'{losses_plot[i]:.2e}', ha='center', va='bottom', fontsize=9, fontweight='bold')

        # (b) Gradient norm vs oracle LR per layer
        ax = axes[1]
        g_norms = [np.linalg.norm(grads[l], 'fro') for l in range(N_LAYERS)]
        ax2 = ax.twinx()
        x_pos = range(N_LAYERS)
        ax.bar([x - 0.15 for x in x_pos], g_norms, width=0.3, color='#4477AA',
               edgecolor='black', label='||G||_F', alpha=0.7)
        ax2.bar([x + 0.15 for x in x_pos], oracle_lrs, width=0.3, color='#CC3311',
                edgecolor='black', label='Oracle LR', alpha=0.7)
        ax.set_xlabel('Layer')
        ax.set_ylabel('Gradient Frobenius Norm', color='#4477AA')
        ax2.set_ylabel('Oracle Learning Rate', color='#CC3311')
        ax.set_xticks(x_pos)
        ax.set_xticklabels([f'Layer {l}' for l in range(N_LAYERS)])
        ax.set_yscale('log')
        ax2.set_yscale('log')
        ax.set_title('Gradient Norms vs Oracle LR')
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plot_path = os.path.join(SCRIPT_DIR, '2_2d_implicit_per_layer_lr.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"\nPlot saved: {plot_path}")
    except ImportError:
        print("\nWARNING: matplotlib not available, skipping plot.")

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions

### What This Experiment Distinguishes

This is a **decomposition experiment**: it separates Muon's advantage into two components:
1. **Scale adaptation** (per-layer LR): the Frobenius normalization in NS
2. **Directional quality** (polar factor): the orthogonalization of the update direction

### Interpretation Guide

- **T1 PASS (oracle matches Muon)**: Muon's advantage under rescaling is *entirely*
  explained by implicit per-layer LR adaptation. The directional content (SV equalization)
  provides no additional value in this regime. This would suggest that Muon's robustness
  to reparametrization is a "simple" consequence of scale normalization.

- **T1 FAIL (oracle cannot match Muon)**: Muon provides value *beyond* per-layer LR.
  Even with perfect scale tuning, the raw gradient direction used by SGD is inferior
  to the orthogonalized direction used by Muon. This supports the gauge-fixing
  interpretation: the polar factor removes gauge degrees of freedom that per-layer
  LR cannot address.

### Quantitative Decomposition

The fraction of Muon's advantage explained by per-layer LR is:
$$\text{fraction explained} = 1 - \frac{R_{\text{oracle}} - 1}{R_{\text{SGD}} - 1}$$
where $R = \text{loss} / \text{Muon loss}$.

### Connection to Other Experiments

- **H3 / H3a**: Normalized SGD vs Muon -- tests the same question without rescaling
- **H6a**: LR confound audit -- checks whether Muon's advantage is an LR artifact
- **Exp 2.2a-c**: Other rescaling robustness tests that this experiment helps explain
- **H15a**: Direction quality metric -- directly measures the directional component